In [1]:
import gcamreader
import pandas as pd
import os
from pathlib import Path
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly.express as px
import matplotlib.pyplot as plt

In [2]:
# =========================================================
# Config
# =========================================================
PROJECT_PATH   = Path("../")
DB_REL_PATH    = PROJECT_PATH / "output"
DB_FILE        = "database_basexdb_korea_2035_v5"
QUERY_FILE     = DB_REL_PATH / "queries" / "Main_queries.xml"

REGION = ['South Korea']

In [3]:
# =========================================================
# DB helpers
# =========================================================
def connect_db():
    return gcamreader.LocalDBConn(DB_REL_PATH, DB_FILE)

def check_query_idx():
    queries = gcamreader.parse_batch_query(os.fspath(QUERY_FILE))
    for i, q in enumerate(queries):
        print(i, q.title)

def run_query(conn, q_idx, scenarios, regions=None):
    queries = gcamreader.parse_batch_query(os.fspath(QUERY_FILE))
    q = queries[q_idx]
    print(q.title)
    df = conn.runQuery(q, scenarios=scenarios, regions=regions)
    df["scenario"] = df["scenario"].str.split(",").str[0]
    return df

def get_scenario_name(conn):
    scenarios = list(conn.listScenariosInDB()['name'].unique())
    return scenarios

In [4]:
def convert_to_mt(row):
    val, unit = row['value'], row['Units']
    if unit == 'Tg':
        return val
    elif unit == 'Gg':
        return val * 1e-3
    elif unit == 'MTC':
        return val * (44.009 / 12.011)
    else:
        raise ValueError(f"Unknown unit: {unit}")

# AR5 100‑yr GWP (no climate–carbon feedbacks)
GWP_AR5 = {
    'CO2':      1,      
    'CH4':     28,      
    'CH4_AGR': 28,
    'CH4_AWB': 28,
    'N2O':    265,      
    'N2O_AGR':265,
    'N2O_AWB':265,
    'HFC125': 3170,     
    'HFC134a':1300,     
    'HFC143a':4800,     
    'HFC23': 12400,     
    'HFC32':   677,     
    'HFC43':  1650,     
    'HFC227ea':3350,    
    'HFC236fa':8060,    
    'SF6':   23500,     
    'C2F6':  11100,     
    'CF4':    6630,     
}


In [5]:
check_query_idx()

0 primary energy consumption by region (avg fossil efficiency)
1 primary energy consumption by region (direct equivalent)
2 primary energy consumption with CCS by region (direct equivalent)
3 resource production
4 resource production by tech and vintage
5 resource supply curves
6 regional primary energy prices
7 elec gen by region (incl CHP)
8 elec gen by subsector
9 elec gen by gen tech
10 elec gen by gen tech and cooling tech
11 elec gen by gen tech and cooling tech and vintage
12 elec gen by gen tech and cooling tech (new)
13 elec energy input by subsector
14 elec energy input by elec gen tech
15 elec energy input by elec gen tech and cooling tech
16 elec prices by sector
17 elec gen costs by subsector
18 elec gen costs by tech
19 elec gen costs by cooling tech
20 elec share-weights by subsector
21 elec share-weights by tech
22 elec share-weights by cooling tech
23 elec td inputs and outputs
24 cogeneration by region
25 elec consumption by demand sector
26 elec sector water withdraw

In [6]:
conn = connect_db()
get_scenario_name(conn)

Database scenarios: High-Ambition-Med, Current-Policies-Med, High-Ambition-Med, Current-Policies-Med, High-Ambition-High, Current-Policies-High, High-Ambition-Med, Current-Policies-Med, Current-Policies-High, High-Ambition-Low, Current-Policies-Low, High-Ambition-Med-AI, Current-Policies-Med-AI, High-Ambition-Med-CPO2040


['High-Ambition-Med',
 'Current-Policies-Med',
 'High-Ambition-High',
 'Current-Policies-High',
 'High-Ambition-Low',
 'Current-Policies-Low',
 'High-Ambition-Med-AI',
 'Current-Policies-Med-AI',
 'High-Ambition-Med-CPO2040']

In [7]:
scenarios = [
    'High-Ambition-Med',
    'Current-Policies-Med',
    'High-Ambition-High',
    'Current-Policies-High',
    'High-Ambition-Low',
    'Current-Policies-Low',
    'High-Ambition-Med-AI',
    'Current-Policies-Med-AI',
    'High-Ambition-Med-CPO2040'
]

In [8]:
df = run_query(conn=conn, q_idx=267, scenarios=scenarios, regions=['South Korea'])
df['sector'].unique()

CO2 sequestration by sector


array(['CO2 removal', 'H2 central production', 'ammonia', 'cement',
       'chemical energy use', 'chemical feedstocks',
       'construction feedstocks', 'elec_coal (IGCC CCS)',
       'elec_coal (conv pul CCS)', 'elec_gas (CC CCS)', 'iron and steel',
       'other industrial feedstocks', 'process heat dac', 'refining',
       'waste biomass for paper'], dtype=object)

In [11]:
df[(df['sector'].isin([
    'H2 central production', 'ammonia', 'cement',
    'chemical energy use', 'elec_coal (IGCC CCS)',
    'elec_coal (conv pul CCS)', 'elec_gas (CC CCS)',
    'iron and steel', 'process heat dac',
    'refining', 'waste biomass for paper'
]))].groupby(['Year', 'scenario'])['value'].sum() * 3.66667

Year  scenario                 
2020  Current-Policies-High         0.173671
      Current-Policies-Low          0.173671
      Current-Policies-Med          0.173671
      Current-Policies-Med-AI       0.173671
      High-Ambition-High            0.173671
      High-Ambition-Low             0.173671
      High-Ambition-Med             0.173671
      High-Ambition-Med-AI          0.173671
      High-Ambition-Med-CPO2040     0.173671
2025  Current-Policies-High         3.870181
      Current-Policies-Low          3.870181
      Current-Policies-Med          3.870181
      Current-Policies-Med-AI       3.870181
      High-Ambition-High            3.870181
      High-Ambition-Low             3.870181
      High-Ambition-Med             3.870181
      High-Ambition-Med-AI          3.870181
      High-Ambition-Med-CPO2040     3.870181
2030  Current-Policies-High         6.185568
      Current-Policies-Low          6.147891
      Current-Policies-Med          6.097034
      Current-Policies-

In [15]:
10.89950 / 9.88319 * 100

110.28321827264274

In [ ]:
12.96070 / 9.88319 * 100

131.13883270482503

In [19]:
14.17780 / 12.96070 * 100

109.39069649015872

In [11]:
df = run_query(conn=conn, q_idx=9, scenarios=scenarios, regions=['South Korea'])
df.groupby(['Year', 'scenario'])['value'].sum() * 277.8

elec gen by gen tech


Year  scenario               
1990  Current-Policies-Med        99.491837
      Current-Policies-Med-AI     99.491837
      High-Ambition-Med           99.491837
      High-Ambition-Med-AI        99.491837
2005  Current-Policies-Med       364.976728
      Current-Policies-Med-AI    364.976728
      High-Ambition-Med          364.976728
      High-Ambition-Med-AI       364.976728
2010  Current-Policies-Med       471.223156
      Current-Policies-Med-AI    471.223156
      High-Ambition-Med          471.223156
      High-Ambition-Med-AI       471.223156
2015  Current-Policies-Med       521.782489
      Current-Policies-Med-AI    521.782489
      High-Ambition-Med          521.782489
      High-Ambition-Med-AI       521.782489
2020  Current-Policies-Med       537.796703
      Current-Policies-Med-AI    537.796703
      High-Ambition-Med          537.796703
      High-Ambition-Med-AI       537.796703
2025  Current-Policies-Med       593.276970
      Current-Policies-Med-AI    593.276970
  

In [27]:
df[(df['sector'].str.contains('resid')) & (df['technology'] == 'gas') & (df['Year'] >= 2015) & (df['Year'] ==2030)]

,Units,scenario,region,sector,subsector,Year,technology,value
426,None Specified,Other-CP,South Korea,resid cooling modern_d1,gas,2030,gas,1.0
494,None Specified,Other-CP,South Korea,resid cooling modern_d10,gas,2030,gas,1.0
562,None Specified,Other-CP,South Korea,resid cooling modern_d2,gas,2030,gas,1.0
630,None Specified,Other-CP,South Korea,resid cooling modern_d3,gas,2030,gas,1.0
698,None Specified,Other-CP,South Korea,resid cooling modern_d4,gas,2030,gas,1.0
766,None Specified,Other-CP,South Korea,resid cooling modern_d5,gas,2030,gas,1.0
834,None Specified,Other-CP,South Korea,resid cooling modern_d6,gas,2030,gas,1.0
902,None Specified,Other-CP,South Korea,resid cooling modern_d7,gas,2030,gas,1.0
970,None Specified,Other-CP,South Korea,resid cooling modern_d8,gas,2030,gas,1.0
1038,None Specified,Other-CP,South Korea,resid cooling modern_d9,gas,2030,gas,1.0


In [68]:
df = run_query(conn=conn, q_idx=122, scenarios=['Other-CP','Other-EA'], regions=['South Korea'])
df[(df['Year'] >= 2025) & (df['Year'] <= 2030)]

iron and steel prices


,Units,scenario,region,sector,Year,value
6,1975$/kg,Other-CP,South Korea,iron and steel,2025,0.165651
7,1975$/kg,Other-CP,South Korea,iron and steel,2030,0.164223
28,1975$/kg,Other-EA,South Korea,iron and steel,2025,0.165653
29,1975$/kg,Other-EA,South Korea,iron and steel,2030,0.185811


In [69]:
0.156661 * 0.13 * 0.5

0.010182965

In [7]:
dfCO2 = run_query(conn=conn, q_idx=262, scenarios=['Other-CP','Other-EA'], regions=['South Korea'])
dfCO2['GHG'] = 'CO2'
dfNonCO2 = run_query(conn=conn, q_idx=270, scenarios=['Other-CP','Other-EA'], regions=['South Korea'])
df = pd.concat([dfCO2, dfNonCO2])
df["emiss(MT)"] = df.apply(convert_to_mt, axis=1)  # uses your utils.convert_to_mt
df["gwpAr5"]    = df["GHG"].map(GWP_AR5).astype(float)
df["MTCO2eq"]   = df["emiss(MT)"] * df["gwpAr5"]
df[(~df['sector'].isin(['trn_aviation_intl', 'trn_shipping_intl']))].groupby(['scenario', 'Year'])['MTCO2eq'].sum()

CO2 emissions by sector (no bio) (excluding resource production)
nonCO2 emissions by sector (excluding resource production)


scenario  Year
Other-CP  1975     54.590644
          1990    295.171616
          2005    588.834536
          2010    694.980605
          2015    750.755896
          2020    702.605474
          2025    689.204194
          2030    636.915075
          2035    558.711188
Other-EA  1975     54.590644
          1990    295.171616
          2005    588.834536
          2010    694.980605
          2015    750.755896
          2020    718.322268
          2025    692.298845
          2030    531.951948
          2035    360.789351
Name: MTCO2eq, dtype: float64

In [10]:
dfCO2 = run_query(conn=conn, q_idx=265, scenarios=['Other-CP','Other-EA'], regions=['South Korea'])
dfCO2['GHG'] = 'CO2'

CO2 emissions by tech (excluding resource production)


In [11]:
dfCO2[(dfCO2['Year'] >= 2035) & (dfCO2['sector'].str.contains('resid'))]

,Units,scenario,region,sector,subsector,technology,Year,value,GHG
764,MTC,Other-CP,South Korea,resid heating coal_d1,coal,coal,2035,0.038586,CO2
776,MTC,Other-CP,South Korea,resid heating coal_d2,coal,coal,2035,0.029975,CO2
784,MTC,Other-CP,South Korea,resid heating coal_d3,coal,coal,2035,0.026305,CO2
792,MTC,Other-CP,South Korea,resid heating coal_d4,coal,coal,2035,0.013154,CO2
831,MTC,Other-CP,South Korea,resid heating modern_d1,gas,gas,2035,0.228668,CO2
...,...,...,...,...,...,...,...,...,...
2505,MTC,Other-EA,South Korea,resid others modern_d5,gas,gas,2035,0.208632,CO2
2520,MTC,Other-EA,South Korea,resid others modern_d6,gas,gas,2035,0.219550,CO2
2535,MTC,Other-EA,South Korea,resid others modern_d7,gas,gas,2035,0.231642,CO2
2550,MTC,Other-EA,South Korea,resid others modern_d8,gas,gas,2035,0.246399,CO2


In [9]:
558.711188 - 38.3

520.411188

In [10]:
1 - 520.411188 / 742.3

0.29892066819345264

In [32]:
dfCO2Map   = pd.read_csv("./extdata/gcamreport/CO2_tech_map.csv", skiprows=[0])
dfNonCO2Map= pd.read_csv("./extdata/gcamreport/nonCO2_emissions_sector_map.csv", skiprows=[0])

In [32]:
df = run_query(conn=conn, q_idx=265, scenarios=['Other-CP','Other-EA'], regions=['South Korea'])
df['sector'].unique()#[(df['Year'] == 2030) & (df['sector'].str.contains('chemical'))]#['value'].sum()

CO2 emissions by tech (excluding resource production)


array(['H2 central production', 'H2 wholesale dispensing',
       'agricultural energy use', 'airCO2', 'ammonia',
       'backup_electricity', 'cement', 'chemical energy use',
       'chemical feedstocks', 'comm cooling', 'comm heating',
       'comm others', 'construction energy use', 'desalinated water',
       'elec_biomass (IGCC)', 'elec_biomass (conv)',
       'elec_coal (IGCC CCS)', 'elec_coal (conv pul CCS)',
       'elec_coal (conv pul)', 'elec_gas (CC CCS)', 'elec_gas (CC)',
       'elec_gas (steam/CT)', 'elec_refined liquids (CC)',
       'elec_refined liquids (steam/CT)', 'electricity', 'gas processing',
       'iron and steel', 'mining energy use',
       'other industrial energy use', 'process heat cement',
       'process heat dac', 'process heat food processing',
       'process heat paper', 'refining', 'regional biomass',
       'regional biomassOil', 'regional corn for ethanol',
       'regional woodpulp for energy', 'resid heating coal_d1',
       'resid heating coal_

In [8]:
df = run_query(conn=conn, q_idx=273, scenarios=['Power-CP'], regions=['South Korea'])
df['sector'].unique()#[(df['Year'] == 2030) & (df['sector'].str.contains('chemical'))]#['value'].sum()

nonCO2 emissions by tech (excluding resource production)


array(['comm cooling', 'electricity_net_ownuse', 'industrial processes',
       'resid cooling modern_d1', 'resid cooling modern_d10',
       'resid cooling modern_d2', 'resid cooling modern_d3',
       'resid cooling modern_d4', 'resid cooling modern_d5',
       'resid cooling modern_d6', 'resid cooling modern_d7',
       'resid cooling modern_d8', 'resid cooling modern_d9',
       'urban processes', 'Beef', 'Corn', 'Dairy', 'FiberCrop',
       'FodderGrass', 'Fruits', 'H2 central production',
       'H2 liquid truck', 'H2 pipeline', 'H2 wholesale dispensing',
       'Legumes', 'MiscCrop', 'NutsSeeds', 'OilCrop', 'OtherGrain',
       'Pork', 'Poultry', 'Rice', 'RootTuber', 'SheepGoat', 'Soybean',
       'UnmanagedLand', 'Vegetables', 'Wheat', 'agricultural energy use',
       'ammonia', 'backup_electricity', 'biomass', 'chemical energy use',
       'comm heating', 'comm others', 'construction energy use',
       'electricity', 'iron and steel', 'mining energy use',
       'other indus

In [9]:
listCrop = [
    'Corn', 'FiberCrop', 'FodderGrass', 'Fruits', 'Legumes', 'MiscCrop', 'NutsSeeds', 
    'OilCrop', 'OtherGrain', 'RootTuber', 'Soybean', 'Vegetables', 'Wheat',
]

In [10]:
df[(df['sector'].isin(listCrop))]['technology'].unique()

array(['CornC4_Korea_RFD_hi', 'CornC4_Korea_RFD_lo',
       'FiberCrop_Korea_IRR_hi', 'FiberCrop_Korea_IRR_lo',
       'FiberCrop_Korea_RFD_hi', 'FiberCrop_Korea_RFD_lo',
       'FodderGrass_Korea_RFD_hi', 'FodderGrass_Korea_RFD_lo',
       'FruitsTree_Korea_IRR_hi', 'FruitsTree_Korea_IRR_lo',
       'FruitsTree_Korea_RFD_hi', 'FruitsTree_Korea_RFD_lo',
       'Fruits_Korea_IRR_hi', 'Fruits_Korea_IRR_lo',
       'Fruits_Korea_RFD_hi', 'Fruits_Korea_RFD_lo',
       'Legumes_Korea_RFD_hi', 'Legumes_Korea_RFD_lo',
       'MiscCrop_Korea_IRR_hi', 'MiscCrop_Korea_IRR_lo',
       'MiscCrop_Korea_RFD_hi', 'MiscCrop_Korea_RFD_lo',
       'NutsSeedsTree_Korea_IRR_hi', 'NutsSeedsTree_Korea_IRR_lo',
       'NutsSeedsTree_Korea_RFD_hi', 'NutsSeedsTree_Korea_RFD_lo',
       'NutsSeeds_Korea_RFD_hi', 'NutsSeeds_Korea_RFD_lo',
       'OilCrop_Korea_IRR_hi', 'OilCrop_Korea_IRR_lo',
       'OilCrop_Korea_RFD_hi', 'OilCrop_Korea_RFD_lo',
       'OtherGrainC4_Korea_RFD_hi', 'OtherGrainC4_Korea_RFD_lo',
 

In [11]:
df["emiss(MT)"] = df.apply(convert_to_mt, axis=1)  # uses your utils.convert_to_mt
df["gwpAr5"]    = df["GHG"].map(GWP_AR5).astype(float)
df["MTCO2eq"]   = df["emiss(MT)"] * df["gwpAr5"]
df[(df['sector'].isin(listCrop))].groupby(['Year', 'scenario'])['MTCO2eq'].sum()

Year  scenario
1975  Power-CP    1.956932
1990  Power-CP    2.296362
2005  Power-CP    2.277849
2010  Power-CP    1.803642
2015  Power-CP    2.001942
2020  Power-CP    2.081022
2025  Power-CP    2.119450
2030  Power-CP    2.152947
2035  Power-CP    2.178591
Name: MTCO2eq, dtype: float64

In [17]:
df[(df['GHG'].str.contains('HFC'))].groupby(['Year', 'scenario'])['MTCO2eq'].sum()

Year  scenario
1990  Power-CP     0.040795
2005  Power-CP    15.341120
2010  Power-CP    23.340671
2015  Power-CP    25.690444
2020  Power-CP    28.499158
2025  Power-CP    30.077220
2030  Power-CP    30.820361
2035  Power-CP    30.305728
Name: MTCO2eq, dtype: float64

In [ ]:
df = run_query(conn=conn, q_idx=166, scenarios=['Power-CP'], regions=['South Korea'])
df['subsector'].unique()#[(df['Year'] == 2030) & (df['sector'].str.contains('chemical'))]#['value'].sum()

In [ ]:
df = run_query(conn=conn, q_idx=166, scenarios=['Power-CP'], regions=['South Korea'])
df['subsector'].unique()#[(df['Year'] == 2030) & (df['sector'].str.contains('chemical'))]#['value'].sum()

In [17]:
df = run_query(conn=conn, q_idx=166, scenarios=['Power-CP'], regions=['South Korea'])
df['subsector'].unique()#[(df['Year'] == 2030) & (df['sector'].str.contains('chemical'))]#['value'].sum()

costs of transport techs


array(['International Aviation', 'Domestic Aviation', 'HSR',
       'Passenger Rail', 'road', 'Bus', 'LDV', '2W and 3W', '4W', 'Car',
       'Large Car and Truck', 'Domestic Ship', 'Freight Rail',
       'Medium truck', 'International Ship'], dtype=object)

In [18]:
dfPol = df[((df['subsector'] == 'Domestic Ship')) & (df['Year'] >= 2025) & (df['Year'] <= 2035)].copy()
# dfPol['subsidy_cp'] = dfPol['value'] * (-0.2)
# dfPol['subsidy_cp'] = dfPol['value'] * (-0.2)
dfPol

,Units,scenario,region,sector,subsector,technology,Year,value
267,1990$/ton-km,Power-CP,South Korea,trn_freight,Domestic Ship,BEV,2025,0.015547
268,1990$/ton-km,Power-CP,South Korea,trn_freight,Domestic Ship,BEV,2030,0.011832
269,1990$/ton-km,Power-CP,South Korea,trn_freight,Domestic Ship,BEV,2035,0.008128
276,1990$/ton-km,Power-CP,South Korea,trn_freight,Domestic Ship,FCEV,2025,0.010209
277,1990$/ton-km,Power-CP,South Korea,trn_freight,Domestic Ship,FCEV,2030,0.009685
278,1990$/ton-km,Power-CP,South Korea,trn_freight,Domestic Ship,FCEV,2035,0.009247
285,1990$/ton-km,Power-CP,South Korea,trn_freight,Domestic Ship,Hybrid Liquids,2025,0.005660
286,1990$/ton-km,Power-CP,South Korea,trn_freight,Domestic Ship,Hybrid Liquids,2030,0.005644
287,1990$/ton-km,Power-CP,South Korea,trn_freight,Domestic Ship,Hybrid Liquids,2035,0.005389
294,1990$/ton-km,Power-CP,South Korea,trn_freight,Domestic Ship,Liquids,2025,0.006466


In [7]:
df = run_query(conn=conn, q_idx=170, scenarios=['Power-CP'], regions=['South Korea'])
df['sector'].unique()#[(df['Year'] == 2030) & (df['sector'].str.contains('chemical'))]#['value'].sum()

transport tech share-weights


array(['trn_aviation_intl', 'trn_freight', 'trn_freight_road', 'trn_pass',
       'trn_pass_road', 'trn_pass_road_LDV', 'trn_pass_road_LDV_4W',
       'trn_shipping_intl'], dtype=object)

In [11]:
df[(df['subsector'] == 'Medium truck') & (df['Year'] >= 2020) & (df['Year'] <= 2035)]

,Units,scenario,region,sector,subsector,Year,technology,value
237,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2020,Hybrid Liquids,0.158869
238,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2020,Liquids,1.000000
239,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2025,BEV,0.034445
240,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2025,FCEV,0.034445
241,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2025,Hybrid Liquids,0.841131
242,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2025,Liquids,1.000000
243,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2030,BEV,0.158869
244,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2030,FCEV,0.158869
245,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2030,Hybrid Liquids,1.000000
246,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2030,Liquids,1.000000


In [10]:
df[(df['sector'] == 'trn_freight_road') & (df['Year'] >= 2020) & (df['Year'] <= 2035)]

,Units,scenario,region,sector,subsector,Year,technology,value
237,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2020,Hybrid Liquids,0.158869
238,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2020,Liquids,1.000000
239,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2025,BEV,0.034445
240,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2025,FCEV,0.034445
241,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2025,Hybrid Liquids,0.841131
242,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2025,Liquids,1.000000
243,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2030,BEV,0.158869
244,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2030,FCEV,0.158869
245,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2030,Hybrid Liquids,1.000000
246,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2030,Liquids,1.000000


In [9]:
df[(df['sector'] == 'trn_pass_road_LDV_4W') & (df['Year'] >= 2020) & (df['Year'] <= 2035)]

,Units,scenario,region,sector,subsector,Year,technology,value
666,None Specified,Power-CP,South Korea,trn_pass_road_LDV_4W,Car,2020,Hybrid Liquids,0.158869
667,None Specified,Power-CP,South Korea,trn_pass_road_LDV_4W,Car,2020,Liquids,1.000000
668,None Specified,Power-CP,South Korea,trn_pass_road_LDV_4W,Car,2020,NG,0.001004
669,None Specified,Power-CP,South Korea,trn_pass_road_LDV_4W,Car,2025,BEV,0.500000
670,None Specified,Power-CP,South Korea,trn_pass_road_LDV_4W,Car,2025,FCEV,0.500000
671,None Specified,Power-CP,South Korea,trn_pass_road_LDV_4W,Car,2025,Hybrid Liquids,0.841131
672,None Specified,Power-CP,South Korea,trn_pass_road_LDV_4W,Car,2025,Liquids,1.000000
673,None Specified,Power-CP,South Korea,trn_pass_road_LDV_4W,Car,2025,NG,0.001004
674,None Specified,Power-CP,South Korea,trn_pass_road_LDV_4W,Car,2030,BEV,1.000000
675,None Specified,Power-CP,South Korea,trn_pass_road_LDV_4W,Car,2030,FCEV,1.000000


In [24]:
df = run_query(conn=conn, q_idx=100, scenarios=['Power-CP'], regions=['South Korea'])
df[(df['Year'] == 2030) & (df['sector'].str.contains('chemical'))]#['value'].sum()

industry final energy by tech and fuel


,Units,scenario,region,sector,subsector,technology,input,Year,value
83,EJ,Power-CP,South Korea,chemical energy use,biomass,biomass,delivered biomass,2030,0.011847
87,EJ,Power-CP,South Korea,chemical energy use,biomass,biomass CCS,delivered biomass,2030,0.000059
95,EJ,Power-CP,South Korea,chemical energy use,coal,coal,delivered coal,2030,0.004963
99,EJ,Power-CP,South Korea,chemical energy use,coal,coal CCS,delivered coal,2030,0.000010
108,EJ,Power-CP,South Korea,chemical energy use,electricity,electricity,elect_td_ind,2030,0.189090
114,EJ,Power-CP,South Korea,chemical energy use,gas,gas,wholesale gas,2030,0.048086
118,EJ,Power-CP,South Korea,chemical energy use,gas,gas CCS,wholesale gas,2030,0.000561
126,EJ,Power-CP,South Korea,chemical energy use,refined liquids,refined liquids,refined liquids industrial,2030,0.022880
130,EJ,Power-CP,South Korea,chemical energy use,refined liquids,refined liquids CCS,refined liquids industrial,2030,0.000352
136,EJ,Power-CP,South Korea,chemical feedstocks,coal,coal,delivered coal,2030,0.024484


In [23]:
df = run_query(conn=conn, q_idx=268, scenarios=['Power-CP'], regions=['South Korea'])
df[(df['Year'] == 2030) & (df['sector'].str.contains('chemical'))]#['value'].sum()

CO2 sequestration by tech


,Units,scenario,region,sector,subsector,technology,Year,value
16,MTC,Power-CP,South Korea,chemical energy use,biomass,biomass CCS,2030,0.001231
20,MTC,Power-CP,South Korea,chemical energy use,coal,coal CCS,2030,0.000237
24,MTC,Power-CP,South Korea,chemical energy use,gas,gas CCS,2030,0.007169
28,MTC,Power-CP,South Korea,chemical energy use,refined liquids,refined liquids CCS,2030,0.006206
37,MTC,Power-CP,South Korea,chemical feedstocks,refined liquids,refined liquids,2030,36.714500


In [13]:
df = run_query(conn=conn, q_idx=321, scenarios=['Power-CP'], regions=['South Korea'])
df#[(df['Year'] == 2020) & (df['subsector'] == 'gas')]#['value'].sum()

costs by tech


,Units,scenario,region,sector,subsector,Year,technology,value
0,$/GJ,Power-CP,South Korea,woodpulp_energy,woodpulp_energy,1975,woodpulp_energy,6.923830
1,$/GJ,Power-CP,South Korea,woodpulp_energy,woodpulp_energy,1990,woodpulp_energy,98.457800
2,$/GJ,Power-CP,South Korea,woodpulp_energy,woodpulp_energy,2005,woodpulp_energy,51.642100
3,$/GJ,Power-CP,South Korea,woodpulp_energy,woodpulp_energy,2010,woodpulp_energy,45.543000
4,$/GJ,Power-CP,South Korea,woodpulp_energy,woodpulp_energy,2015,woodpulp_energy,26.855700
...,...,...,...,...,...,...,...,...
6175,1990$/ton-km,Power-CP,South Korea,trn_shipping_intl,International Ship,2030,Liquids,0.002578
6176,1990$/ton-km,Power-CP,South Korea,trn_shipping_intl,International Ship,2035,BEV,0.004874
6177,1990$/ton-km,Power-CP,South Korea,trn_shipping_intl,International Ship,2035,FCEV,0.004116
6178,1990$/ton-km,Power-CP,South Korea,trn_shipping_intl,International Ship,2035,Hybrid Liquids,0.002262


In [14]:
df[(df['sector'] == 'carbon-storage')]

,Units,scenario,region,sector,subsector,Year,technology,value
5982,1990$/tC,Power-CP,South Korea,carbon-storage,offshore carbon-storage,1975,offshore carbon-storage,212.00
5983,1990$/tC,Power-CP,South Korea,carbon-storage,offshore carbon-storage,1990,offshore carbon-storage,212.00
5984,1990$/tC,Power-CP,South Korea,carbon-storage,offshore carbon-storage,2005,offshore carbon-storage,212.00
5985,1990$/tC,Power-CP,South Korea,carbon-storage,offshore carbon-storage,2010,offshore carbon-storage,212.00
5986,1990$/tC,Power-CP,South Korea,carbon-storage,offshore carbon-storage,2015,offshore carbon-storage,212.00
5987,1990$/tC,Power-CP,South Korea,carbon-storage,offshore carbon-storage,2020,offshore carbon-storage,212.00
5988,1990$/tC,Power-CP,South Korea,carbon-storage,offshore carbon-storage,2025,offshore carbon-storage,212.00
5989,1990$/tC,Power-CP,South Korea,carbon-storage,offshore carbon-storage,2030,offshore carbon-storage,212.00
5990,1990$/tC,Power-CP,South Korea,carbon-storage,offshore carbon-storage,2035,offshore carbon-storage,212.00
5991,1990$/tC,Power-CP,South Korea,carbon-storage,onshore carbon-storage,1975,onshore carbon-storage,1.00


In [25]:
df = run_query(conn=conn, q_idx=11, scenarios=['Power-CP'], regions=['South Korea'])
df[(df['Year'] == 2020) & (df['subsector'] == 'gas')]#['value'].sum()

elec gen by gen tech and cooling tech and vintage


,Units,scenario,region,sector,subsector,technology,output,Year,value
224,EJ,Power-CP,South Korea,electricity,gas,"gas (CC) (dry cooling),year=2015",elec_gas (CC),2020,0.012519
228,EJ,Power-CP,South Korea,electricity,gas,"gas (CC) (dry cooling),year=2020",elec_gas (CC),2020,0.001774
242,EJ,Power-CP,South Korea,electricity,gas,"gas (CC) (once through),year=2015",elec_gas (CC),2020,0.002087
250,EJ,Power-CP,South Korea,electricity,gas,"gas (CC) (recirculating),year=2015",elec_gas (CC),2020,0.270339
254,EJ,Power-CP,South Korea,electricity,gas,"gas (CC) (recirculating),year=2020",elec_gas (CC),2020,0.067489
268,EJ,Power-CP,South Korea,electricity,gas,"gas (CC) (seawater),year=2015",elec_gas (CC),2020,0.068611
272,EJ,Power-CP,South Korea,electricity,gas,"gas (CC) (seawater),year=2020",elec_gas (CC),2020,0.018227
286,EJ,Power-CP,South Korea,electricity,gas,"gas (steam/CT) (dry cooling),year=2015",elec_gas (steam/CT),2020,0.000078
290,EJ,Power-CP,South Korea,electricity,gas,"gas (steam/CT) (dry cooling),year=2020",elec_gas (steam/CT),2020,0.000016
304,EJ,Power-CP,South Korea,electricity,gas,"gas (steam/CT) (once through),year=2015",elec_gas (steam/CT),2020,0.000010


In [17]:
5260 / 4333

1.2139395338102932

In [18]:
5920 / 4333

1.3662589429956151

In [20]:
5530 / 4333

1.2762520193861067

In [22]:
3090 / 4333

0.7131317793676437